# Exemplos MQL com sample_mflix (PyMongo)

Notebook com exemplos práticos das lições do material aplicados ao database `sample_mflix`. Conexão configurada para MongoDB local em `mongodb://localhost:27017/`.

## Requisitos e conexão

Instale dependências se necessário e conecte ao MongoDB local.

In [3]:
!pip install pymongo dnspython
from pymongo import MongoClient
from pprint import pprint

# Conexão local (MongoDB em localhost:27017)
client = MongoClient("mongodb://localhost:27017/")
db = client.Sample_flix

movies = db.movies
comments = db.comments
users = db.users

print("Conectado a", db.name)
print("Collections (exemplo):", db.list_collection_names()[:10])

Conectado a Sample_flix
Collections (exemplo): ['sessions', 'theaters', 'movies', 'comments', 'users']



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Operação Read: find(), find_one(), projeções e cursores

In [4]:
# 1.1 Encontrar um filme por título
doc = movies.find_one({"title": "Back to the Future"})
pprint(doc)

{'_id': ObjectId('573a1398f29313caabce9682'),
 'awards': {'nominations': 24,
            'text': 'Won 1 Oscar. Another 18 wins & 24 nominations.',
            'wins': 19},
 'cast': ['Michael J. Fox',
          'Christopher Lloyd',
          'Lea Thompson',
          'Crispin Glover'],
 'countries': ['USA'],
 'directors': ['Robert Zemeckis'],
 'fullplot': 'Marty McFly, a typical American teenager of the Eighties, is '
             'accidentally sent back to 1955 in a plutonium-powered DeLorean '
             '"time machine" invented by slightly mad scientist. During his '
             'often hysterical, always amazing trip back in time, Marty must '
             'make certain his teenage parents-to-be meet and fall in love - '
             'so he can get back to the future.',
 'genres': ['Adventure', 'Comedy', 'Sci-Fi'],
 'imdb': {'id': 88763, 'rating': 8.5, 'votes': 636511},
 'languages': ['English'],
 'lastupdated': '2015-09-12 00:29:36.890000000',
 'metacritic': 86,
 'plot': 'A young

In [7]:
# 1.2 Buscar vários documentos com filtro simples e projeção
cursor = movies.find({"year": {"$gte": 2015}}, {"title": 1, "year": 1}).sort("year", -1).limit(5)
for m in cursor:
    pprint(m)

{'_id': ObjectId('573a13e6f29313caabdc6a9a'),
 'title': 'The Masked Saint',
 'year': 2016}
{'_id': ObjectId('573a13b5f29313caabd44e2b'), 'title': 'Ant-Man', 'year': 2015}
{'_id': ObjectId('573a13b8f29313caabd4d540'),
 'title': 'The Danish Girl',
 'year': 2015}
{'_id': ObjectId('573a13b1f29313caabd3719d'),
 'title': 'The Stanford Prison Experiment',
 'year': 2015}
{'_id': ObjectId('573a13adf29313caabd2b765'),
 'title': 'Jurassic World',
 'year': 2015}


In [8]:
# 1.3 Uso de operadores de comparação e conjunto ($in)
cursor = movies.find({"year": {"$gt": 2000}, "genres": {"$in": ["Drama", "Comedy"]}}, {"title":1, "genres":1, "year":1}).limit(10)
for m in cursor:
    pprint(m)

{'_id': ObjectId('573a1393f29313caabcdcb42'),
 'genres': ['Comedy', 'Fantasy', 'Romance'],
 'title': 'Kate & Leopold',
 'year': 2001}
{'_id': ObjectId('573a1398f29313caabceb1fe'),
 'genres': ['Drama'],
 'title': 'Crime and Punishment',
 'year': 2002}
{'_id': ObjectId('573a139af29313caabcf0562'),
 'genres': ['Drama'],
 'title': 'What Is It?',
 'year': 2005}
{'_id': ObjectId('573a139af29313caabcf0718'),
 'genres': ['Drama', 'Music', 'Romance'],
 'title': 'Glitter',
 'year': 2001}
{'_id': ObjectId('573a139af29313caabcf0809'),
 'genres': ['Crime', 'Drama', 'History'],
 'title': 'The Manson Family',
 'year': 2003}
{'_id': ObjectId('573a139af29313caabcf0869'),
 'genres': ['Drama', 'Thriller', 'Crime'],
 'title': 'The Dancer Upstairs',
 'year': 2002}
{'_id': ObjectId('573a139af29313caabcf0ea6'),
 'genres': ['Biography', 'Drama', 'Romance'],
 'title': 'Frida',
 'year': 2002}
{'_id': ObjectId('573a139af29313caabcf0ec0'),
 'genres': ['Comedy', 'Fantasy', 'Romance'],
 'title': 'Kate & Leopold',
 

## 2. Operadores lógicos: $and, $or, $not, $nor

In [ ]:
# 2.1 $and implícito (filtros combinados)
res = movies.find_one({"year": {"$gte": 2000}, "rated": "PG-13"})
pprint(res)

In [ ]:
# 2.2 $or
cursor = movies.find({"$or": [{"rated": "R"}, {"rated": "PG-13"}]}, {"title":1, "rated":1}).limit(10)
for m in cursor:
    pprint(m)

In [ ]:
# 2.3 $not (negando uma condição)
cursor = movies.find({"year": {"$not": {"$gte": 2010}}}, {"title":1, "year":1}).limit(5)
for m in cursor:
    pprint(m)

## 3. Create: insert_one() e insert_many() (exemplo em coleção de teste)

In [ ]:
test_coll = db.get_collection("students_example")
# limpar demo (cuidado em produção)
test_coll.delete_many({})

# 3.1 insertOne
res = test_coll.insert_one({"name":"Ana", "score": 85})
print("InsertedId:", res.inserted_id)
pprint(test_coll.find_one({"name":"Ana"}))

In [ ]:
# 3.2 insertMany
res = test_coll.insert_many([
    {"name":"Bruno", "score": 90},
    {"name":"Carla", "score": 78}
], ordered=False)
print("InsertedIds:", res.inserted_ids)
for s in test_coll.find({}):
    pprint(s)

## 4. Update: update_one(), update_many(), operadores de atualização ($set, $inc, $currentDate, $rename, $addToSet, $push)

In [ ]:
# Preparar documentos de exemplo para os exercícios de update
test_coll.delete_many({})
test_coll.insert_many([
    {"name":"Davi", "milk": 5, "tags": ["morning"]},
    {"name":"Elisa", "milk": 7, "tags": ["evening"]},
    {"name":"Felipe", "milk": 3}
])
for s in test_coll.find({}):
    pprint(s)

In [ ]:
# 4.1 updateOne com $set
res = test_coll.update_one({"name":"Davi"}, {"$set": {"milk": 6}})
print("matched:", res.matched_count, "modified:", res.modified_count)
pprint(test_coll.find_one({"name":"Davi"}))

In [ ]:
# 4.2 updateMany com $inc
res = test_coll.update_many({}, {"$inc": {"milk": 1}})
print("matched:", res.matched_count, "modified:", res.modified_count)
for s in test_coll.find({}):
    pprint(s)

In [ ]:
# 4.3 $currentDate e $rename
res = test_coll.update_one({"name":"Elisa"}, {"$currentDate":{"lastModified": True}, "$rename": {"milk":"milk_amount"}})
pprint(test_coll.find_one({"name":"Elisa"}))

In [ ]:
# 4.4 Array updates: $addToSet e $push / $each
res = test_coll.update_one({"name":"Felipe"}, {"$addToSet": {"tags": "afternoon"}})
res = test_coll.update_one({"name":"Davi"}, {"$push": {"tags": {"$each": ["dairy", "farm"]}}})
for s in test_coll.find({}):
    pprint(s)

## 5. Delete: delete_one(), delete_many(), remover todos com filtro {} e drop()

In [ ]:
# 5.1 Excluir um documento
res = test_coll.delete_one({"name":"Bruno"})
print("deletedCount:", res.deleted_count)
for s in test_coll.find({}):
    pprint(s)

In [ ]:
# 5.2 Excluir vários
res = test_coll.delete_many({"milk": {"$lt": 5}})
print("deletedCount:", res.deleted_count)
for s in test_coll.find({}):
    pprint(s)

## 6. Consultas com $expr (comparar campos no mesmo documento)

In [ ]:
# Preparar coleção cow_demo
cow = db.get_collection("cow_demo")
cow.delete_many({})
for c in range(1,10):
    expected = c-1 if c%2==0 else c+1
    cow.insert_one({"name":"daisy", "milk": c, "expected_milk": expected})
print('Documentos inseridos:', cow.count_documents({}))

In [ ]:
# 6.1 Usar $expr para comparar campos (resultado)
cursor = cow.find({"$expr": {"$gt": ["$milk", "$expected_milk"]}}, {"milk":1, "expected_milk":1})
for d in cursor:
    pprint(d)
print("count:", cow.count_documents({"$expr": {"$gt": ["$milk", "$expected_milk"]}}))

## 7. Consultas em arrays e $elemMatch, projeção com $slice e operador posicional $

In [ ]:
arrc = db.get_collection("array_demo")
arrc.delete_many({})
arrc.insert_one({
    "title":"array test",
    "scores":[ {"type":"quiz","score":8}, {"type":"exam","score":6}, {"type":"homework","score":9} ]
})
print('Documento array_demo inserido:')
pprint(arrc.find_one({}))

In [ ]:
# 7.1 $elemMatch para encontrar arrays que contêm elemento com condições
doc = arrc.find_one({"scores": {"$elemMatch": {"type":"exam", "score": {"$lt": 7}}}})
pprint(doc)

In [ ]:
# 7.2 Projeção: retornar apenas primeiro elemento do array usando $slice
doc = arrc.find_one({}, {"scores": {"$slice": 1}})
pprint(doc)

In [ ]:
# 7.3 Operador posicional $ para projetar o primeiro elemento do array que casou com o filtro
doc = arrc.find_one({"scores.score": {"$gte": 9}}, {"scores.$": 1})
pprint(doc)

## 8. Agregações básicas (pipeline)

In [ ]:
# 8.1 Exemplo: contagem de filmes por ano (pipeline simples)
pipeline = [
    {"$match": {"year": {"$gte": 2000}}},
    {"$group": {"_id": "$year", "count": {"$sum": 1}}},
    {"$sort": {"_id": 1}},
    {"$limit": 10}
]
for doc in movies.aggregate(pipeline):
    pprint(doc)

In [ ]:
# 8.2 Exemplo: top 5 filmes com maior número de comentários
pipeline = [
    {"$lookup": {"from": "comments", "localField": "_id", "foreignField": "movie_id", "as": "movie_comments"}},
    {"$project": {"title": 1, "num_comments": {"$size": "$movie_comments"}}},
    {"$sort": {"num_comments": -1}},
    {"$limit": 5}
]
for doc in movies.aggregate(pipeline):
    pprint(doc)

## 9. Boas práticas e limpeza

- Use índices adequados para melhorar performance.
- Teste atualizações e deleções primeiro com filtros restritos.
- Em produção, configure writeConcern e autenticação.
- Limpar coleções de exemplo após os testes se necessário.

In [ ]:
# 9.1 Células de limpeza (opcional) -- descomente se quiser apagar as coleções de exemplo
# db.students_example.drop()
# db.cow_demo.drop()
# db.array_demo.drop()
print("Limpeza opcional: descomente as linhas se quiser apagar as coleções de exemplo.")